In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/processed/unsw_nb15_processed.csv")

In [3]:
df['is_attack'] = df['label']

In [4]:
df[['srcip', 'dstip']].head()

,srcip,dstip
0,0.378964,-1.828500
1,0.546825,0.689279
2,-1.131788,-1.532290
3,0.043241,-1.680395
4,0.211102,0.985488


In [5]:
threat_intel_db = {
    "175.45.176.2": {"reputation": "malicious", "confidence": 90},
    "185.220.101.1": {"reputation": "malicious", "confidence": 85},
    "91.240.118.172": {"reputation": "malicious", "confidence": 60},
    "10.0.0.5": {"reputation": "internal", "confidence":0}
}

In [6]:
def lookup_ip_reputation(ip):
    if ip in threat_intel_db:
        return threat_intel_db[ip]['reputation'], threat_intel_db[ip]['confidence']
    else:
        return "unknown", 0

In [7]:
df[['src_reputation', 'src_confidence']] = df['dstip'].apply(
    lambda x: pd.Series(lookup_ip_reputation(x))
)

In [8]:
df[['srcip', 'src_reputation', 'src_confidence', 'label']].head(10)

,srcip,src_reputation,src_confidence,label
0,0.378964,unknown,0,0
1,0.546825,unknown,0,0
2,-1.131788,unknown,0,1
3,0.043241,unknown,0,0
4,0.211102,unknown,0,0
5,0.378964,unknown,0,0
6,-0.124620,unknown,0,0
7,-0.963926,unknown,0,1
8,1.218270,unknown,0,0
9,0.378964,unknown,0,0


In [9]:
def calculate_risk(row):
    risk = 0

    if row['label'] == 1:
        risk += 50

    if row['src_reputation'] == 'malicious':
        risk += row['src_confidence']
    elif row['src_reputation'] == 'suspicious':
        risk += 30

    return min(risk, 100)

In [10]:
df['risk_score'] = df.apply(calculate_risk, axis=1)

In [11]:
def classify_severity(score):
    if score >= 80:
        return "Critical"
    elif score >= 50:
        return "High"
    elif score >= 20:
        return "Medium"
    else:
        return "Low"

In [12]:
df['alert_severity'] = df['risk_score'].apply(classify_severity)

In [13]:
df[['srcip', 'label', 'src_reputation', 'risk_score', 'alert_severity']].sample(10)

,srcip,label,src_reputation,risk_score,alert_severity
7359,0.378964,0,unknown,0,Low
2928,0.882547,0,unknown,0,Low
7367,0.378964,0,unknown,0,Low
1177,0.714686,0,unknown,0,Low
1096,0.043241,0,unknown,0,Low
6670,-0.292481,0,unknown,0,Low
566,0.882547,0,unknown,0,Low
8172,0.714686,0,unknown,0,Low
5875,0.043241,0,unknown,0,Low
6540,0.882547,0,unknown,0,Low


In [14]:
df.to_csv("../data/processed/unsw_nb15_enriched.csv", index=False)